In [1]:
# ======================================================
# XGBoost 自动调参脚本（只调4个参数，每个模型独立随机种子）
# ======================================================

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_score
from scipy.stats import randint, uniform
import warnings

warnings.filterwarnings("ignore")

# ======================================================
# 1. 路径与变量设置
# ======================================================

data_path = r"E:/excel/FILES/博士申请/第二篇论文/data/小区all_cleaned999_deduplicated3.xlsx"

independent_vars = [
    "NEWTCC", "price", "FAR_new",
    "NEWBCI", "BAH", "SVF_MN", "jungong",
    "LSI", "COHESION", "PD",
    "TCC180M", "BCI180M", "BAH180M"
]

lst_vars = [
    "lst0603",
    "lst0803",
    "lst0619",
    "lst0719",
    "lst0609",
    "lst0724"
]

# ======================================================
# 2. 读取数据
# ======================================================

df = pd.read_excel(data_path)

# ======================================================
# 3. 定义要搜索的超参数
# ======================================================

param_dist = {
    "n_estimators": randint(100, 2001),
    "learning_rate": uniform(0.01, 0.29),
    "max_depth": randint(2, 9),
    "min_child_weight": randint(1, 10)
}

# ======================================================
# 4. 循环训练 6 个模型
# ======================================================

all_results = []

for i, current_lst in enumerate(lst_vars):
    print("\n" + "=" * 60)
    print(f"Training XGBoost model for: {current_lst}")
    print("=" * 60)

    # 准备数据
    selected_vars = independent_vars + [current_lst]
    model_data = df[selected_vars].dropna()
    X = model_data[independent_vars]
    y = model_data[current_lst]

    # 不同模型使用不同随机种子
    seed = 123 + i
    cv = KFold(n_splits=5, shuffle=True, random_state=seed)

    xgb_base = XGBRegressor(
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1,
        tree_method="hist"
    )

    # 随机搜索
    search = RandomizedSearchCV(
        estimator=xgb_base,
        param_distributions=param_dist,
        n_iter=120,  # 搜索次数可增加，覆盖范围更大
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=seed,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X, y)

    best_model = search.best_estimator_
    best_params = search.best_params_

    # 5 折交叉验证 RMSE 和 R²
    rmse_scores = -cross_val_score(
        best_model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )
    r2_scores = cross_val_score(
        best_model, X, y,
        scoring="r2",
        cv=cv,
        n_jobs=-1
    )

    mean_rmse = rmse_scores.mean()
    sd_rmse = rmse_scores.std()
    mean_r2 = r2_scores.mean()
    sd_r2 = r2_scores.std()

    print(f"\nBest parameters for {current_lst}:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    print(f"\n5-fold CV RMSE: {mean_rmse:.4f} ± {sd_rmse:.4f}")
    print(f"5-fold CV R²:   {mean_r2:.4f} ± {sd_r2:.4f}")

    # 保存结果
    result_row = {
        "LST": current_lst,
        "CV_RMSE_mean": mean_rmse,
        "CV_RMSE_sd": sd_rmse,
        "CV_R2_mean": mean_r2,
        "CV_R2_sd": sd_r2,
        **best_params
    }
    all_results.append(result_row)

# ======================================================
# 5. 汇总并导出 CSV
# ======================================================

results_df = pd.DataFrame(all_results)

out_path = r"E:/excel/FILES/博士申请/第二篇论文/xgboost_best_params_6_models.xlsx"
results_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("\nAll done! Best parameter table saved to:")
print(out_path)


Training XGBoost model for: lst0603
Fitting 5 folds for each of 120 candidates, totalling 600 fits

Best parameters for lst0603:
  learning_rate: 0.010262322314032808
  max_depth: 4
  min_child_weight: 8
  n_estimators: 280

5-fold CV RMSE: 1.0251 ± 0.0462
5-fold CV R²:   0.5325 ± 0.0470

Training XGBoost model for: lst0803
Fitting 5 folds for each of 120 candidates, totalling 600 fits

Best parameters for lst0803:
  learning_rate: 0.013554440302736896
  max_depth: 3
  min_child_weight: 7
  n_estimators: 1034

5-fold CV RMSE: 1.2097 ± 0.0857
5-fold CV R²:   0.5950 ± 0.0569

Training XGBoost model for: lst0619
Fitting 5 folds for each of 120 candidates, totalling 600 fits

Best parameters for lst0619:
  learning_rate: 0.03841486617391577
  max_depth: 4
  min_child_weight: 6
  n_estimators: 173

5-fold CV RMSE: 1.1248 ± 0.0359
5-fold CV R²:   0.5963 ± 0.0242

Training XGBoost model for: lst0719
Fitting 5 folds for each of 120 candidates, totalling 600 fits

Best parameters for lst0719:


In [2]:
# ======================================================
# XGBoost 自带 TreeSHAP：
# 计算 Mean Absolute SHAP Value + 相对重要性 + 绘图
# ======================================================

import os
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBRegressor
import shap
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# ======================================================
# 1. 文件路径
# ======================================================

data_path = r"E:/excel/FILES/博士申请/第二篇论文/data/小区all_cleaned999_deduplicated2.xlsx"
param_path = r"E:/excel/FILES/博士申请/第二篇论文/xgboost_best_params_6_models.xlsx"
plot_dir = r"E:/excel/FILES/博士申请/第二篇论文/plot/plots"

os.makedirs(plot_dir, exist_ok=True)

# ======================================================
# 2. 读取数据
# ======================================================

df = pd.read_excel(data_path)
param_df = pd.read_excel(param_path)

# ======================================================
# 3. 自变量列表
# ======================================================

independent_vars = [
    "NEWTCC", "price", "FAR_new",
    "NEWBCI", "BAH", "SVF_MN", "jungong",
    "LSI", "COHESION", "PD",
    "TCC180M", "BCI180M", "BAH180M"
]

# 变量显示名称，可选
var_name_map = {
    "NEWTCC": "TCC",
    "price": "House Price",
    "FAR_new": "FAR",
    "NEWBCI": "BCI",
    "BAH": "BAH",
    "SVF_MN": "BVF",
    "jungong": "CY",
    "LSI": "LSI",
    "COHESION": "COHESION",
    "PD": "PD",
    "TCC180M": "TCC_buffer",
    "BCI180M": "BCI_buffer",
    "BAH180M": "BAH_buffer"
}

display_vars = [var_name_map[v] for v in independent_vars]

# ======================================================
# 4. 循环训练每个 LST 模型并计算 SHAP
# ======================================================

all_importance = []

for _, row in param_df.iterrows():

    current_lst = row["LST"]

    print("\n" + "=" * 70)
    print(f"Processing: {current_lst}")
    print("=" * 70)

    selected_vars = independent_vars + [current_lst]
    model_data = df[selected_vars].dropna()

    X = model_data[independent_vars].copy()
    y = model_data[current_lst].copy()

    # 每个模型使用其对应的最佳参数
    learning_rate = float(row["learning_rate"])
    max_depth = int(row["max_depth"])
    min_child_weight = int(row["min_child_weight"])
    n_estimators = int(row["n_estimators"])

    print(
        f"Using params: "
        f"n_estimators={n_estimators}, "
        f"learning_rate={learning_rate}, "
        f"max_depth={max_depth}, "
        f"min_child_weight={min_child_weight}"
    )

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        random_state=123,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(X, y)

    # ==================================================
    # XGBoost 自带 TreeSHAP
    # pred_contribs=True 返回：
    # [每个变量的SHAP值 + 最后一列bias/base value]
    # ==================================================

    dmatrix = xgb.DMatrix(X, feature_names=independent_vars)

    shap_values_full = model.get_booster().predict(
        dmatrix,
        pred_contribs=True
    )

    shap_values = shap_values_full[:, :-1]   # 去掉最后一列 bias
    base_value = shap_values_full[:, -1]     # bias/base value

    # 检查维度
    print("X shape:", X.shape)
    print("SHAP shape:", shap_values.shape)

    # ==================================================
    # Mean Absolute SHAP Value
    # ==================================================

    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    tmp = pd.DataFrame({
        "LST": current_lst,
        "Variable": independent_vars,
        "Variable_Label": display_vars,
        "MeanAbsSHAP": mean_abs_shap
    })

    tmp["RelativeImportance_percent"] = (
        tmp["MeanAbsSHAP"] / tmp["MeanAbsSHAP"].sum()
    ) * 100

    tmp = tmp.sort_values("MeanAbsSHAP", ascending=False)

    all_importance.append(tmp)

    print(tmp[["Variable", "MeanAbsSHAP", "RelativeImportance_percent"]])

    # ==================================================
    # 5. 相对重要性柱状图：y轴按重要性降序
    # ==================================================

       # ==================================================
    # Mean Absolute SHAP Importance Plot
    # ==================================================
    
    plot_df = tmp.sort_values(
        "MeanAbsSHAP",
        ascending=True
    )
    
    fig, ax = plt.subplots(
        figsize=(6,6)
    )
    
    bars = ax.barh(
        plot_df["Variable_Label"],
        plot_df["MeanAbsSHAP"],
        height=0.7
    )
    
    # -------------------------------
    # 论文风格
    # -------------------------------
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)
    
    ax.tick_params(
        axis="both",
        direction="out",
        length=4,
        width=1,
        labelsize=15
    )
    
    # -------------------------------
    # 坐标轴标签
    # -------------------------------
    
    ax.set_xlabel(
        "Mean Absolute SHAP Value",
        fontsize=15,
        fontweight="bold"
    )
    
    ax.set_ylabel("")
    
    ax.set_title(
        current_lst,
        fontsize=15,
        fontweight="bold"
    )
    
    # -------------------------------
    # 自动扩展X轴
    # 防止标签被截断
    # -------------------------------
    
    max_val = plot_df["MeanAbsSHAP"].max()
    
    ax.set_xlim(
        0,
        max_val * 1.18
    )
    
    # -------------------------------
    # 数值标签
    # -------------------------------
    
    for bar in bars:
    
        width = bar.get_width()
    
        ax.text(
            width + max_val * 0.02,
            bar.get_y() + bar.get_height()/2,
            f"{width:.2f}",
            va="center",
            ha="left",
            fontsize=15
        )
    
    # -------------------------------
    # 去除网格
    # -------------------------------
    
    ax.grid(False)
    
    # -------------------------------
    # 布局
    # -------------------------------
    
    plt.tight_layout()
    
    plt.savefig(
        os.path.join(
            plot_dir,
            f"{current_lst}_MeanAbsSHAP.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )
    
    plt.close()

    # ==================================================
    # 6. SHAP summary plot
    # 这里使用 XGBoost 自带算出的 shap_values
    # ==================================================

    X_display = X.copy()
    X_display.columns = display_vars
    
    # 使用 plot_size 控制图大小
    shap.summary_plot(
        shap_values,
        X_display,
        plot_size=(8, 6),  # 图比例
        show=False
    )
    
    # 获取当前坐标轴
    ax = plt.gca()
    
    # -------------------------------
    # 修改坐标轴字体大小 + 加粗
    # -------------------------------
    ax.tick_params(
        axis='x',       # 横轴
        labelsize=14,   # 字体大小
        width=2,        # 刻度线宽度
        labelcolor='black',
        direction='out'
    )
    ax.tick_params(
        axis='y',       # 纵轴
        labelsize=12,
        width=2,
        labelcolor='black',
        direction='out'
    )
    
    # -------------------------------
    # 修改坐标轴 spine 线条粗细
    # -------------------------------
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['top'].set_linewidth(0)    # 可以隐藏
    ax.spines['right'].set_linewidth(0)  # 可以隐藏
    
    # -------------------------------
    # 标题加粗
    # -------------------------------
    plt.title(
        f"{current_lst} - SHAP Summary Plot",
        fontsize=16,
        fontweight='bold'
    )
    
    plt.tight_layout()
    
    out_summary = os.path.join(
        plot_dir,
        f"{current_lst}_XGB_SHAP_summary_plot.png"
    )
    
    plt.savefig(out_summary, dpi=600, bbox_inches="tight")
    plt.close()

# ======================================================
# 7. 保存所有模型的重要性结果
# ======================================================

importance_df = pd.concat(all_importance, ignore_index=True)

out_csv = r"E:/excel/FILES/博士申请/第二篇论文/XGB_SHAP_MeanAbsImportance.csv"
out_xlsx = r"E:/excel/FILES/博士申请/第二篇论文/XGB_SHAP_MeanAbsImportance.xlsx"

importance_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
importance_df.to_excel(out_xlsx, index=False)

print("\n" + "=" * 70)
print("All done.")
print("Saved:")
print(out_csv)
print(out_xlsx)
print(plot_dir)
print("=" * 70)


Processing: lst0603
Using params: n_estimators=258, learning_rate=0.0653920170574221, max_depth=3, min_child_weight=6
X shape: (1312, 13)
SHAP shape: (1312, 13)
    Variable  MeanAbsSHAP  RelativeImportance_percent
4        BAH     0.335099                   16.751350
11   BCI180M     0.270051                   13.499683
10   TCC180M     0.251119                   12.553268
6    jungong     0.242067                   12.100772
0     NEWTCC     0.230545                   11.524760
1      price     0.179388                    8.967467
3     NEWBCI     0.147352                    7.366019
2    FAR_new     0.104922                    5.244966
12   BAH180M     0.066874                    3.342975
7        LSI     0.057193                    2.859062
8   COHESION     0.043679                    2.183496
9         PD     0.039738                    1.986467
5     SVF_MN     0.032401                    1.619707

Processing: lst0803
Using params: n_estimators=1034, learning_rate=0.013554440302

In [8]:
import xgboost
import shap

print(xgboost.__version__)
print(shap.__version__)

3.2.0
0.48.0


In [18]:
from PIL import Image

# ==================================================
# 拼接函数
# ==================================================

def combine_two_figures(
        left_img,
        right_img,
        output_img
):

    img1 = Image.open(left_img)
    img2 = Image.open(right_img)

    width = img1.width + img2.width
    height = max(img1.height, img2.height)

    combined = Image.new(
        "RGB",
        (width, height),
        (255,255,255)
    )

    combined.paste(
        img1,
        (0,0)
    )

    combined.paste(
        img2,
        (img1.width,0)
    )

    combined.save(
        output_img,
        dpi=(600,600)
    )

# ==================================================
# 0603 + 0803
# ==================================================

combine_two_figures(

    os.path.join(
        plot_dir,
        "lst0603_MeanAbsSHAP.png"
    ),

    os.path.join(
        plot_dir,
        "lst0803_MeanAbsSHAP.png"
    ),

    os.path.join(
        plot_dir,
        "0603_0803_SHAP.png"
    )
)

# ==================================================
# 0619 + 0719
# ==================================================

combine_two_figures(

    os.path.join(
        plot_dir,
        "lst0619_MeanAbsSHAP.png"
    ),

    os.path.join(
        plot_dir,
        "lst0719_MeanAbsSHAP.png"
    ),

    os.path.join(
        plot_dir,
        "0619_0719_SHAP.png"
    )
)

# ==================================================
# 0609 + 0724
# ==================================================

combine_two_figures(

    os.path.join(
        plot_dir,
        "lst0609_MeanAbsSHAP.png"
    ),

    os.path.join(
        plot_dir,
        "lst0724_MeanAbsSHAP.png"
    ),

    os.path.join(
        plot_dir,
        "0609_0724_SHAP.png"
    )
)

print("\nFigure combinations completed.")


Figure combinations completed.


In [19]:
# ==================================================
# 三张拼图再纵向拼接
# ==================================================

img1 = Image.open(
    os.path.join(
        plot_dir,
        "0603_0803_SHAP.png"
    )
)

img2 = Image.open(
    os.path.join(
        plot_dir,
        "0619_0719_SHAP.png"
    )
)

img3 = Image.open(
    os.path.join(
        plot_dir,
        "0609_0724_SHAP.png"
    )
)

# 最终画布尺寸
final_width = max(
    img1.width,
    img2.width,
    img3.width
)

final_height = (
    img1.height +
    img2.height +
    img3.height
)

final_img = Image.new(
    "RGB",
    (final_width, final_height),
    (255, 255, 255)
)

# 依次粘贴
y_offset = 0

for img in [img1, img2, img3]:

    final_img.paste(
        img,
        (0, y_offset)
    )

    y_offset += img.height

# 保存
final_img.save(
    os.path.join(
        plot_dir,
        "All_SHAP_Importance.png"
    ),
    dpi=(600,600)
)

print(
    "\nFinal figure saved:",
    os.path.join(
        plot_dir,
        "All_SHAP_Importance.png"
    )
)


Final figure saved: E:/excel/FILES/博士申请/第二篇论文/plot/plots\All_SHAP_Importance.png
